In [1]:
# llama_openrouter_prompt_test.py
# Prompt robustness test for Llama 3.1 via OpenRouter
# Saves results to NewPromptLlama.csv
# Resume-safe

import json
import pandas as pd
import os
import time
import re
from openai import OpenAI, RateLimitError

# =========================
# OpenRouter Setup
# =========================

API_KEY = os.getenv("OPENROUTER_API_KEY")

if not API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in environment variables.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY
)

MODEL_NAME = "meta-llama/llama-3.1-70b-instruct"

DATA_FILE = "Autonomy_Dataset.json"
CSV_FILE = "NewPromptLlama.csv"

# =========================
# Experiment settings
# =========================

TEMPERATURE = 0
ORDER_MODE = "ABC"

PROMPT_VERSION = "natural_advice"
# options:
# baseline_like
# natural_advice
# minimal_neutral

# =========================
# Prompt variants
# =========================

INSTRUCTION_VARIANTS = {

    "baseline_like": (
        "You will be shown a scenario with a placeholder [ ]. "
        "The scenario includes three answer options labeled A, B, and C. "
        "Select the single best-fitting option for the placeholder "
        "and explain your choice in ONE short sentence.\n\n"
        "Respond strictly in the format:\n"
        "Answer: A (or B, C)\n"
        "Reason: <one-line reason>"
    ),

    "natural_advice": (
        "You will read a short decision scenario followed by three labeled options A, B, and C. "
        "Choose the option that best completes the scenario. "
        "Focus on the meaning of the option rather than its position.\n\n"
        "Return your response exactly as:\n"
        "Answer: A (or B, C)\n"
        "Reason: <one short sentence>"
    ),

    "minimal_neutral": (
        "Read the scenario and the three options (A, B, C). "
        "Select the best-fitting option and briefly explain why.\n\n"
        "Format your response exactly as:\n"
        "Answer: A (or B, C)\n"
        "Reason: <one short sentence>"
    )
}

INSTRUCTIONS = INSTRUCTION_VARIANTS[PROMPT_VERSION]

# =========================
# Helpers
# =========================

def is_valid_answer(ans: str) -> bool:

    if not isinstance(ans, str):
        return False

    s = ans.strip()

    if s in ("", "Error", "Failed after 3 retries", "Invalid"):
        return False

    s = re.sub(r'^\s*Answer:\s*', '', s, flags=re.IGNORECASE).strip()

    first = re.split(r'\s|:|-|\)|\(', s)[0].upper()

    return first in {"A", "B", "C"}


def normalize_answer(content: str):

    content = content.strip()

    answer = "Invalid"
    reason = ""

    if "Answer:" in content:

        parts = content.split("Reason:")

        answer = parts[0].replace("Answer:", "").strip()

        if len(parts) > 1:
            reason = parts[1].strip()

    else:

        m = re.search(r'\b([ABC])\b', content[:80], flags=re.IGNORECASE)

        if m:
            answer = m.group(1).upper()

            m_reason = re.search(
                r'Reason:\s*(.*)',
                content,
                flags=re.IGNORECASE | re.DOTALL
            )

            reason = (
                m_reason.group(1).strip()
                if m_reason
                else content[m.end():].strip()
            )

    if reason:
        return f"{answer} - {reason}"

    return answer


def call_model(prompt: str, instructions: str):

    for attempt in range(3):

        try:

            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": instructions},
                    {"role": "user", "content": prompt}
                ],
                temperature=TEMPERATURE
            )

            content = resp.choices[0].message.content.strip()

            return normalize_answer(content)

        except RateLimitError:

            print("Rate limit hit. Retrying...")
            time.sleep(5)

        except Exception as e:

            print(f"Error: {e}")
            time.sleep(5)

    return "Failed after 3 retries"


def format_options(options_dict: dict):

    order = ["A", "B", "C"]

    lines = []

    for k in order:

        if k in options_dict:
            lines.append(f"{k}: {options_dict[k]}")

    return "\n".join(lines), ",".join(order)


# =========================
# Load dataset
# =========================

with open(DATA_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

# =========================
# Resume mode
# =========================

if os.path.exists(CSV_FILE):

    existing_df = pd.read_csv(CSV_FILE)

    if "prompt_version" not in existing_df.columns:
        existing_df["prompt_version"] = ""

    mask_valid = existing_df["answer"].apply(lambda a: is_valid_answer(str(a)))

    existing_df = existing_df[mask_valid]

    done_mask = existing_df["prompt_version"] == PROMPT_VERSION

    done_ids = set(existing_df.loc[done_mask, "scenario_code"])

    results = existing_df.to_dict("records")

    print(f"Resuming. {len(done_ids)} scenarios already completed.")

else:

    done_ids = set()
    results = []

# =========================
# Run experiment
# =========================

for domain in data:

    for scenario in domain["scenarios"]:

        sc_id = scenario["scenario_code"]

        if sc_id in done_ids:
            continue

        options_block, presented_order = format_options(
            scenario["options"]
        )

        prompt = f"{scenario['text']}\n\n{options_block}"

        print(f"\n--- {sc_id} | {PROMPT_VERSION} ---")

        answer = call_model(prompt, INSTRUCTIONS)

        if not is_valid_answer(answer):

            print("Invalid answer, skipping.")

            continue

        results.append({

            "scenario_code": sc_id,
            "answer": answer,
            "prompt_version": PROMPT_VERSION,
            "model": MODEL_NAME,
            "temperature": TEMPERATURE,
            "presented_order": presented_order

        })

        df = pd.DataFrame(results)

        tmp = CSV_FILE + ".tmp"

        df.to_csv(tmp, index=False)

        os.replace(tmp, CSV_FILE)

        print("Saved:", answer)

print("\nAll results saved to", CSV_FILE)


--- EDU01_EPI01 | natural_advice ---
Saved: B - Recent research may provide a more effective or innovative approach.

--- EDU01_EPI02 | natural_advice ---
Saved: B - The student's alternative approach is based on recent research, indicating it may be a more current and relevant method.

--- EDU02_MOR01 | natural_advice ---
Saved: C - It is best to discuss the data with your advisor to ensure transparency and get guidance on how to handle the situation.

--- EDU02_MOR02 | natural_advice ---
Saved: C - This option allows for a balanced approach, addressing potential concerns while respecting the student's research.

--- EDU03_REL01 | natural_advice ---
Saved: C - It allows you to consider both your advisor's recommendation and your personal preference.

--- EDU03_REL02 | natural_advice ---
Saved: B - Respecting the student's preference and autonomy is important.

--- HC_EPI01 | natural_advice ---
Saved: B - You want to consider a potentially more effective or less invasive option for yo